In [1]:
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import sqlite3

#DATA SCRAPING

In [2]:
product_names=[]
Product_price=[]
Rating=[]
Available_stock=[]
categories=[]
no_of_categories=["travel_2/index.html","mystery_3/index.html","historical-fiction_4/index.html","sequential-art_5/index.html"]
for i in no_of_categories:
 response1=requests.get(f"https://books.toscrape.com/catalogue/category/books/{str(i)}")
 response1.encoding="utf-8"
 soup=BeautifulSoup(response1.text,"html.parser")
 titles=soup.find("ol",class_="row")
 names=titles.find_all("article",class_="product_pod")
 for book in names:
  title=book.find("h3").find("a").get("title")
  product_names.append(title)
  price=book.find("div",class_="product_price").find("p",class_="price_color").text
  Product_price.append(price)
  rating=book.find("p").get("class")[1]
  Rating.append(rating)
  stock=book.find("p",class_="instock availability")
  if stock:
   stocking=stock.get_text(strip=True)
   Available_stock.append(stocking)
  else:
   Available_stock.append("out_of_the_stock")
  category=soup.find("div",class_="col-sm-8 col-md-9").find("div",class_="page-header").text.strip()
  categories.append(category)

#DATA CLEANING

In [3]:
df1=pd.DataFrame({"title":product_names,"price":Product_price,"star_rating":Rating,"availability":Available_stock,"category":categories})
df1["price"]=df1["price"].str.replace("£","")
df1["price_gbp"]=pd.to_numeric(df1["price"])
df1=df1.drop(columns=["price"])
df1["star_rating"]=df1["star_rating"].map({"One":1,"Two":2,"Three":3,"Four":4,"Five":5})
df1["rating"]=df1["star_rating"]
df1=df1.drop(columns=["star_rating"])
df1["availability"]=df1["availability"].map({"In stock":True,"not_in_stock":False})
df1["price_inr"]=df1["price_gbp"]*105.5
df1.loc[df1["category"] == "Travel", "category_id"] = 101
df1.loc[df1["category"] == "Mystery", "category_id"] = 102
df1.loc[df1["category"] == "Historical Fiction", "category_id"] = 103
df1.loc[df1["category"] == "Sequential Art", "category_id"] = 104
df1["book_id"]=range(len(df1["title"]))
df2_1=df1["category_id"].unique()
df2_2=df1["category"].unique()
df2_3=pd.DataFrame({"category_id":df2_1,"category":df2_2})
df2_4=np.array(df2_3)
df3=df1[["book_id","title","price_gbp","price_inr","rating","availability","category_id"]]
df3_3=np.array(df3)

#CREATE DATABASE AND INSERT DATA

In [4]:
with sqlite3.connect("books_store.db") as p:
  cursor=p.cursor()
  cursor.execute('''
  CREATE TABLE categories(
  category_id INT,
  category_name TEXT)
  ''')
  cursor.execute('''
  CREATE TABLE books(
  book_id INT,
  title TEXT,
  price_gbp REAL,
  price_inr REAL,
  rating INT,
  in_stock INT,
  category_id INT)
  ''')
  p.commit()

In [5]:
with sqlite3.connect("books_store.db") as p:
  cursor=p.cursor()
  cursor.executemany('''
  INSERT INTO categories(category_id,category_name)
  VALUES(?,?)''',df2_4)
  cursor.executemany('''
  INSERT INTO books(book_id,title,price_gbp,price_inr,rating,in_stock,category_id)
  VALUES(?,?,?,?,?,?,?)''',df3_3)

#SQL queries

In [6]:
#first 5 rows of table
c=pd.read_sql("SELECT * FROM books LIMIT 5",p)
print(c)

   book_id                                              title  price_gbp  \
0        0                            It's Only the Himalayas      45.17   
1        1  Full Moon over Noah’s Ark: An Odyssey to Mount...      49.43   
2        2  See America: A Celebration of Our National Par...      48.87   
3        3  Vagabonding: An Uncommon Guide to the Art of L...      36.94   
4        4                               Under the Tuscan Sun      37.33   

   price_inr  rating  in_stock  category_id  
0   4765.435       2         1          101  
1   5214.865       4         1          101  
2   5155.785       3         1          101  
3   3897.170       2         1          101  
4   3938.315       3         1          101  


In [7]:
#unique rating
c=pd.read_sql("SELECT DISTINCT rating FROM books",p)
print(c)

   rating
0       2
1       4
2       3
3       1
4       5


In [8]:
#5 star rating books and price_inr
c=pd.read_sql("SELECT title,price_inr FROM books WHERE rating=5",p)
print(c)


                                                title  price_inr
0                  1,000 Places to See Before You Die   2751.440
1              A Time of Torment (Charlie Parker #14)   5100.925
2   What Happened on Beale Street (Secrets of the ...   2676.535
3   The Bachelor Girl's Guide to Murder (Herringfo...   5517.650
4             A Flight of Arrows (The Pathfinders #2)   5858.415
5                                        Mrs. Houdini   3191.375
6                               The Passion of Dolssa   2987.760
7                              Voyager (Outlander #3)   2222.885
8                                        The Red Tent   3762.130
9   Scott Pilgrim's Precious Little Life (Scott Pi...   5516.595
10  Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...   1435.855


In [9]:
#top 5 costly books and rating
c=pd.read_sql("SELECT title,rating,price_inr FROM books ORDER BY price_inr DESC LIMIT 5",p)
print(c)

                                     title  rating  price_inr
0            Boar Island (Anna Pigeon #19)       3   6275.140
1         A Year in Provence (Provence #1)       4   6000.840
2                      The Past Never Ends       4   5960.750
3         The Last Painting of Sara de Vos       2   5860.525
4  A Flight of Arrows (The Pathfinders #2)       5   5858.415


In [10]:
#less rating and high cost of books(5000 Rs)
c=pd.read_sql("SELECT title,rating,price_inr FROM books WHERE rating=1 AND price_inr>5000",p)
print(c)

                                               title  rating  price_inr
0                                 Tipping the Velvet       1   5669.570
1  The Guernsey Literary and Potato Peel Pie Society       1   5225.415
2  orange: The Complete Collection 1 (orange: The...       1   5107.255


In [11]:
#books price is 1000 to 2000 Rs
c=pd.read_sql("SELECT title,price_inr FROM books WHERE price_inr BETWEEN 1000 AND 2000",p)
print(c)

                                                title  price_inr
0                                    A Murder in Time   1755.520
1              That Darkness (Gardiner and Renner #1)   1468.560
2                Tastes Like Fear (DI Marnie Rome #3)   1127.795
3             A Study in Scarlet (Sherlock Holmes #1)   1765.015
4                          Hide Away (Eve Duncan #20)   1249.120
5                                   Playing with Fire   1446.405
6                                         Lilac Girls   1823.040
7          The Constant Princess (The Tudor Court #1)   1753.410
8   Tsubasa: WoRLD CHRoNiCLE 2 (Tsubasa WoRLD CHRo...   1717.540
9   Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...   1435.855
10                                           Patience   1071.880
11  Outcast, Vol. 1: A Darkness Surrounds Him (Out...   1628.920
12                                      Camp Midnight   1801.940


In [12]:
#each book which category belongs to
c=pd.read_sql("SELECT b.title,c.category_name FROM books b LEFT JOIN categories c ON b.category_id=c.category_id ",p)
print(c)

                                                title   category_name
0                             It's Only the Himalayas          Travel
1   Full Moon over Noah’s Ark: An Odyssey to Mount...          Travel
2   See America: A Celebration of Our National Par...          Travel
3   Vagabonding: An Uncommon Guide to the Art of L...          Travel
4                                Under the Tuscan Sun          Travel
..                                                ...             ...
66                       I am a Hero Omnibus Volume 1  Sequential Art
67               Giant Days, Vol. 2 (Giant Days #5-8)  Sequential Art
68                               Danganronpa Volume 1  Sequential Art
69  Codename Baboushka, Volume 1: The Conclave of ...  Sequential Art
70                                      Camp Midnight  Sequential Art

[71 rows x 2 columns]


#pd.read_sql() and pd.merge() ,checking output equal.

In [13]:
#which category has highest price_inr(pd.read_sql())
c=pd.read_sql("SELECT b.title,c.category_name,b.price_inr FROM books b LEFT JOIN categories c ON b.category_id=c.category_id ORDER BY price_inr DESC LIMIT 5",p)
print(c)

                                     title       category_name  price_inr
0            Boar Island (Anna Pigeon #19)             Mystery   6275.140
1         A Year in Provence (Provence #1)              Travel   6000.840
2                      The Past Never Ends             Mystery   5960.750
3         The Last Painting of Sara de Vos  Historical Fiction   5860.525
4  A Flight of Arrows (The Pathfinders #2)  Historical Fiction   5858.415


In [14]:
#in "travel" category of highest rating book (pd.read_sql())
c=pd.read_sql("""SELECT b.title,c.category_name,b.price_inr,b.rating FROM books b LEFT JOIN categories c ON b.category_id=c.category_id WHERE category_name="Travel" ORDER BY rating DESC LIMIT 1 """,p)
print(c)

                                title category_name  price_inr  rating
0  1,000 Places to See Before You Die        Travel    2751.44       5


In [15]:
books_df=pd.read_sql("SELECT * FROM books",p)
categories_df=pd.read_sql("SELECT * FROM categories",p)


In [16]:
#which category has highest price_inr(pd.merge())
c = pd.merge(books_df,categories_df,on="category_id",how="left")
c=c[["title","price_inr","category_name"]]
c=c.sort_values("price_inr",ascending=False)
print(c.head(5))

                                      title  price_inr       category_name
25            Boar Island (Anna Pigeon #19)   6275.140             Mystery
7          A Year in Provence (Provence #1)   6000.840              Travel
13                      The Past Never Ends   5960.750             Mystery
48         The Last Painting of Sara de Vos   5860.525  Historical Fiction
33  A Flight of Arrows (The Pathfinders #2)   5858.415  Historical Fiction


In [17]:
#in "travel" category of highest rating book (pd.merge())
c = pd.merge(books_df,categories_df,on="category_id",how="left")
c=c[["title","category_name","price_inr","rating"]]
c=c[c["category_name"]=="Travel"]
c=c.sort_values("rating",ascending=False)
print(c.head(1))

                                 title category_name  price_inr  rating
10  1,000 Places to See Before You Die        Travel    2751.44       5
